In [1]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from ipyleaflet import Map, Marker, Polyline
from ipywidgets import Output
from IPython.display import display
from geopy.geocoders import Nominatim
import heapq


In [2]:
# Fonction de géolocalisation inverse : Faire une recherche d’adresse inversée : à partir d’une latitude et d’une longitude, 
def reverse_geocode(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True, language='fr') #Appelle la méthode .reverse() pour effectuer le reverse geocoding
        return location.address if location else "Adresse inconnue"
    except:
        return "Erreur lors de la géolocalisation"

In [3]:
# Fonction de traitement du clic sur la carte
def handle_map_click(**kwargs): 
    if kwargs.get('type') == 'click': 
        lat, lon = kwargs['coordinates'] 
        marker = Marker(location=(lat, lon)) 
        m.add_layer(marker) 
        clicks.append((lat, lon)) 
        
        
        if len(clicks) == 2: 
            origin_coords = clicks[0] 
            dest_coords = clicks[1] 
            calculate_and_display_route(origin_coords, dest_coords)
            

In [4]:
# Implémentation de Dijkstra
def dijkstra_manual(graph, start_node, end_node):
    distances = {node: float('inf') for node in graph.nodes()}
    previous_nodes = {}
    distances[start_node] = 0
    queue = [(0, start_node)]

    while queue:
        current_distance, current_node = heapq.heappop(queue)

        if current_node == end_node:
            break

        for neighbor in graph.neighbors(current_node):
            edge_data = graph.get_edge_data(current_node, neighbor)
            weight = edge_data[0]['length'] if edge_data else float('inf')
            distance = current_distance + weight

            if distance < distances[neighbor]:
                distances[neighbor] = distance
                previous_nodes[neighbor] = current_node
                heapq.heappush(queue, (distance, neighbor))

    # Reconstruire le chemin
    path = []
    current = end_node
    while current != start_node:
        path.insert(0, current)
        current = previous_nodes.get(current)
        if current is None:
            return None  # Pas de chemin
    path.insert(0, start_node)
    return path

In [5]:
# Fonction pour calculer et afficher le chemin
def calculate_and_display_route(origin_coords, dest_coords): 
    orig_node = ox.distance.nearest_nodes(G, origin_coords[1], origin_coords[0]) 
    dest_node = ox.distance.nearest_nodes(G, dest_coords[1], dest_coords[0])
    if orig_node == dest_node:
        print("Les points sélectionnés sont identiques.")
        return
    try:
        shortest_path = dijkstra_manual(G, orig_node, dest_node) 
        if shortest_path is None: 
            print("Aucun chemin trouvé.")
            return
        route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in shortest_path]
        route_line = Polyline(locations=route_coords, color="red", fill=False) 
        m.add_layer(route_line) 
        total_distance = 0    # Calcul de la distance
        for i in range(len(shortest_path) - 1):
            node1 = shortest_path[i]
            node2 = shortest_path[i + 1]
            weight = G[node1][node2][0]['length']
            total_distance += weight
        origin_name = reverse_geocode(origin_coords[0], origin_coords[1])
        dest_name = reverse_geocode(dest_coords[0], dest_coords[1])
        print(f"\n Chemin calculé entre les points :")
        print(f"📍 Départ: {origin_name}")
        print(f"📍 Arrivée: {dest_name}")
        print("Nœuds et longueurs d'arêtes :")
        for i in range(len(shortest_path) - 1):
            node1 = shortest_path[i]
            node2 = shortest_path[i + 1]
            weight = G[node1][node2][0]['length']
            print(f"  - Nœud {node1} → {node2} | {weight:.2f} m")
        print(f"Distance totale : {total_distance:.2f} m")
        plot_reduced_graph(shortest_path)
    except Exception as e:
        print(f"Erreur lors du calcul du chemin : {e}")
        

In [6]:
# Affichage du graphe réduit
def plot_reduced_graph(shortest_path):  
    nodes_to_plot = set(shortest_path) 
    for node in shortest_path:
        neighbors = list(G.neighbors(node))
        nodes_to_plot.update(neighbors)
        
        for neighbor in neighbors:
            neighbors_of_neighbor = list(G.neighbors(neighbor))
            nodes_to_plot.update(neighbors_of_neighbor)

    subgraph = G.subgraph(nodes_to_plot).copy()
    plt.figure(figsize=(10, 8))  
    
    

    for u, v in subgraph.edges():  
        x = [G.nodes[u]['x'], G.nodes[v]['x']]  
        y = [G.nodes[u]['y'], G.nodes[v]['y']]
        plt.plot(x, y, color='gray', alpha=0.5) 

        weight = G[u][v][0]['length']  
        mid_x = (x[0] + x[1]) / 2
        mid_y = (y[0] + y[1]) / 2
        plt.text(mid_x, mid_y, f'{weight:.1f}m', fontsize=6, ha='center') 


    for node in shortest_path:  
        x, y = G.nodes[node]['x'], G.nodes[node]['y'] 
        plt.scatter(x, y, color='blue') 
    for i in range(len(shortest_path) - 1):
        u = shortest_path[i]
        v = shortest_path[i + 1]
        x = [G.nodes[u]['x'], G.nodes[v]['x']]
        y = [G.nodes[u]['y'], G.nodes[v]['y']]
        plt.plot(x, y, color='red', linewidth=3)
        

    plt.title("Chemin trouvé avec djikstra")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()


In [7]:
# Initialisation du géocodeur
geolocator = Nominatim(user_agent="my_osmnx_app")




In [8]:
# Télécharger le graphe routier autour de Tunis
tunis_point = (36.8065, 10.1815) #Définit le point central à Tunis.
G = ox.graph_from_point(tunis_point, dist=5000, network_type='drive') #Télécharge un graphe routier (type drive) dans un rayon de 5 km autour du point.

# Initialisation de la carte interactive
m = Map(center=tunis_point, zoom=13) #m est une carte centrée sur Tunis.
clicks = [] #clicks est une liste vide qui va stocker les clics de l’utilisateur.



In [9]:
# Activation des clics
m.on_interaction(handle_map_click)  #Connecte la carte à la fonction de clic.
display(m) #Affiche la carte interactive 

No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe
No such comm: e4eb7fc3681347ec8c90f13a26c3d0fe


Map(center=[36.8065, 10.1815], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…